# LiteLLM on Amazon Bedrock, hands-on

**Day 9 · Agentic AI Bootcamp**

Every cell below runs against Amazon Bedrock with no extra infrastructure. You only need AWS credentials with Bedrock access and the Claude Haiku 4.5 model enabled in `us-east-1`.

What you will run:
- unified `completion()` on Bedrock, streaming, the temperature+top_p gotcha
- fallbacks, retries, cost tracking, structured output, routing / load balancing
- embeddings (the seed of the RAG notebook)
- LiteLLM driving Strands, LangChain (`ChatLiteLLM`), and LangGraph
- an AgentCore entrypoint written to disk for deployment

Every API call here is real. Nothing is mocked.

## How to run

**VS Code**
1. Create and select a venv: `python -m venv .venv` then pick it as the kernel.
2. Set credentials: `aws configure` (or export `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`).
3. Run the install cell, then run top to bottom.

**Google Colab**
1. Run the install cell.
2. Set credentials via `os.environ["AWS_ACCESS_KEY_ID"] = ...` and `AWS_SECRET_ACCESS_KEY`, or Colab secrets.
3. Run top to bottom.

**What changes in production:** use an IAM role instead of access keys, read region and model from config (no hardcoding), keep `num_retries` + `fallbacks` on, log `completion_cost` per call, and add tracing (Langfuse / MLflow).

In [1]:
%pip install -q litellm "strands-agents[litellm]" langchain-litellm langgraph langchain-core numpy boto3

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, json, litellm

REGION = "us-east-1"
os.environ["AWS_REGION_NAME"] = REGION            # LiteLLM reads this for Bedrock
os.environ.setdefault("AWS_DEFAULT_REGION", REGION)

# Chat model as a LiteLLM Bedrock string. The us. inference-profile prefix is mandatory
# for on-demand Claude on Bedrock; a naked "anthropic.claude-..." will fail.
MODEL = "bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Embedding model (used later here and reused in the RAG notebook)
EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"

# Bedrock Claude rejects requests that send BOTH temperature and top_p.
# Drop unsupported params globally so no framework default trips it.
litellm.drop_params = True

# Credential sanity check. Bedrock uses AWS credentials, NOT an api_key.
import boto3
try:
    ident = boto3.client("sts", region_name=REGION).get_caller_identity()
    print("AWS credentials OK. Account:", ident["Account"])
except Exception as e:
    print("AWS credentials NOT found. Fix this before running the cells below.")
    print("VS Code: aws configure  (or export AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)")
    print("Colab:   set os.environ[...] or use Colab secrets")
    print("Detail:", e)

AWS credentials OK. Account: 452203592848


## 1. The core call: one function, any model

`completion()` is the whole surface for most work. Change the model string and the same call reaches any provider.

In [2]:
resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is Amazon Bedrock?"}],
    max_tokens=200,
)
answer = resp.choices[0].message.content
print(answer)
assert answer and len(answer) > 0
print("\n[OK] Unified completion works. Response shape is OpenAI-style regardless of provider.")

Amazon Bedrock is a fully managed AWS service that provides access to foundation models from various providers through a single API, enabling developers to build generative AI applications without managing infrastructure.

[OK] Unified completion works. Response shape is OpenAI-style regardless of provider.


## 2. The gotcha that bites everyone

Bedrock Claude fails if a request carries both `temperature` and `top_p`. Many frameworks default `top_p=1.0`. With `litellm.drop_params = True` (set in config), the unsupported param is removed instead of raising.

In [3]:
resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
    temperature=0.3,
    top_p=0.9,      # would conflict on Bedrock Claude; dropped safely by drop_params
    max_tokens=50,
)
print(resp.choices[0].message.content)
print("[OK] drop_params handled the temperature + top_p conflict.")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: BedrockException - {"message":"The model returned the following errors: `temperature` and `top_p` cannot both be specified for this model. Please use only one."}

## 3. Streaming

Token-by-token deltas, same shape across providers.

In [4]:
print("Streaming: ", end="")
for chunk in litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
    max_tokens=60,
):
    piece = chunk.choices[0].delta.content or ""
    print(piece, end="", flush=True)
print("\n[OK] Streaming deltas received.")

Streaming: 1
2
3
4
5
[OK] Streaming deltas received.


## 4. Fallbacks

Primary model fails, LiteLLM retries the next model automatically. Here the primary is a deliberately invalid model id, so it fails and the good backup answers. That reliably exercises the fallback path.

In [5]:
from litellm import Router

router = Router(
    model_list=[
        {"model_name": "primary",
         "litellm_params": {"model": "bedrock/us.anthropic.claude-nonexistent-v1:0",
                            "aws_region_name": REGION}},
        {"model_name": "backup",
         "litellm_params": {"model": MODEL, "aws_region_name": REGION}},
    ],
    fallbacks=[{"primary": ["backup"]}],
)

resp = router.completion(
    model="primary",
    messages=[{"role": "user", "content": "Reply with a short greeting."}],
    max_tokens=50,
)
print(resp.choices[0].message.content)
print("[OK] Primary failed, fallback answered. This is uptime insurance for throttling and outages.")

Hello! How can I help you today?
[OK] Primary failed, fallback answered. This is uptime insurance for throttling and outages.


## 5. Retries and timeouts

Transient errors (throttling, network) get retried with backoff before reaching your code. This is the standard cushion for the 424 / throttling errors seen at concurrent load.

In [6]:
resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    num_retries=3,
    timeout=30,
    max_tokens=10,
)
print(resp.choices[0].message.content)
print("[OK] Retries and timeout configured.")

ready
[OK] Retries and timeout configured.


## 6. Cost tracking

Token usage priced per model, no manual math.

In [7]:
resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": "One fun fact about octopuses."}],
    max_tokens=120,
)
print(resp.choices[0].message.content)

cost = litellm.completion_cost(completion_response=resp)
u = resp.usage
print(f"\nTokens in/out: {u.prompt_tokens}/{u.completion_tokens}")
print(f"Estimated cost: ${cost:.6f}")
print("[OK] Per-call cost available. Pair with an Application Inference Profile for per-project billing.")

# Octopuses Have Nine Brains

Two-thirds of an octopus's neurons are located in their **arms**, not their head. This means each arm can essentially think and act semi-independently—an octopus can have one arm exploring and solving a puzzle while another arm is hunting, all without conscious direction from the central brain. It's like having nine mini-brains working in parallel!

Tokens in/out: 15/89
Estimated cost: $0.000506
[OK] Per-call cost available. Pair with an Application Inference Profile for per-project billing.


## 7. Structured output

Force machine-readable output for pipelines. Below we prompt for JSON and parse it, which always works. The native path is `response_format={"type": "json_object"}`, which LiteLLM maps to tool use on Bedrock Claude.

In [8]:
prompt = ("Return ONLY valid JSON, no prose, with keys landmark, city, country "
          "for the Eiffel Tower.")
resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=120,
)
raw = resp.choices[0].message.content.strip()
raw = raw.replace("```json", "").replace("```", "").strip()   # strip fences if present
data = json.loads(raw)
print(data)
assert {"landmark", "city", "country"}.issubset(data.keys())
print("[OK] Parsed structured JSON.")

{'landmark': 'Eiffel Tower', 'city': 'Paris', 'country': 'France'}
[OK] Parsed structured JSON.


## 8. Routing and load balancing

Several deployments under one name. The Router spreads traffic and reroutes on failure. Callers only ever say the alias.

In [9]:
from litellm import Router

lb = Router(model_list=[
    {"model_name": "claude", "litellm_params": {"model": MODEL, "aws_region_name": REGION}},
    {"model_name": "claude", "litellm_params": {"model": MODEL, "aws_region_name": REGION}},
])

for i in range(2):
    r = lb.completion(model="claude",
                      messages=[{"role": "user", "content": f"Reply with only the number {i}."}],
                      max_tokens=10)
    print(f"call {i}:", r.choices[0].message.content)
print("[OK] Router balanced calls across deployments named 'claude'.")

call 0: 0
call 1: 1
[OK] Router balanced calls across deployments named 'claude'.


## 9. Embeddings: the seed of RAG

Same unified interface for embedding models. Nearest text to a query is found by cosine similarity. This is exactly what the RAG notebook builds on.

In [10]:
import numpy as np

texts = ["refund and money-back policy", "how to reset your password", "office lunch menu"]
emb = litellm.embedding(model=EMBED_MODEL, input=texts)
vecs = [np.array(d["embedding"], dtype=float) for d in emb.data]
print("Embedding dimension:", len(vecs[0]))

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

q = np.array(litellm.embedding(model=EMBED_MODEL, input=["get my money back"]).data[0]["embedding"])
for t, v in zip(texts, vecs):
    print(f"{cosine(q, v):.3f}  <-  {t}")
print("[OK] The refund text should score highest. Meaning became geometry.")

Embedding dimension: 1024
0.414  <-  refund and money-back policy
0.095  <-  how to reset your password
0.022  <-  office lunch menu
[OK] The refund text should score highest. Meaning became geometry.


## 10. LiteLLM with Strands

Strands ships a LiteLLM model provider, so the same Bedrock string now drives a Strands agent with tools. Swap `model_id` and the loop runs on any provider.

In [11]:
from strands import Agent, tool
from strands.models.litellm import LiteLLMModel

@tool
def percent_of(part: float, whole: float) -> float:
    """Return part percent of whole. Example: percent_of(15, 240)."""
    return whole * part / 100.0

model = LiteLLMModel(
    model_id=MODEL,
    params={"max_tokens": 600, "temperature": 0.3},
)
agent = Agent(model=model, tools=[percent_of],
              system_prompt="Use the percent_of tool for percentage questions.")
result = agent("What is 15 percent of 240?")
print(result)
print("[OK] Strands agent ran on a LiteLLM model with a tool.")


Tool #1: percent_of
15 percent of 240 is **36**.15 percent of 240 is **36**.

[OK] Strands agent ran on a LiteLLM model with a tool.


## 11. LiteLLM with LangChain

`ChatLiteLLM` is a standard LangChain chat model. It supports `bind_tools` and slots into LCEL chains.

**Known bug (Jan 2026):** `bind_tools().astream()` on Bedrock can misroute to an OpenAI endpoint and fail. Use `invoke` for tool-using Bedrock agents, or `langchain-aws` `ChatBedrockConverse` for native streaming with tools.

In [12]:
from langchain_litellm import ChatLiteLLM
from langchain_core.tools import tool as lc_tool

llm = ChatLiteLLM(model=MODEL, temperature=0.3, max_tokens=300)
print("invoke:", llm.invoke("Name three uses of vector embeddings.").content)

@lc_tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

llm_with_tools = llm.bind_tools([multiply])
ai = llm_with_tools.invoke("What is 6 times 7? Use the multiply tool.")   # invoke, NOT astream
print("tool_calls:", ai.tool_calls)
print("[OK] ChatLiteLLM invoke + bind_tools works (non-streaming path).")

/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/Day-10 - Strands/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


invoke: # Three Uses of Vector Embeddings

1. **Semantic Search and Retrieval**
   - Convert text, images, or other data into vectors to find semantically similar items
   - Used in search engines, recommendation systems, and document retrieval to match queries with relevant content based on meaning rather than exact keyword matches

2. **Machine Learning Feature Representation**
   - Transform raw data (like words, images, or categorical variables) into numerical vectors that machine learning models can process
   - Enables models to learn patterns and relationships in the data more effectively than working with raw inputs

3. **Similarity and Clustering Analysis**
   - Calculate distances between embeddings to measure how similar two items are
   - Used for clustering similar items together, detecting duplicates, or identifying related content without explicit labeling

These applications are foundational to modern AI systems, including large language models, recommendation engines, 

## 12. LiteLLM with LangGraph

LangGraph is the wiring; LiteLLM is the model inside a node. Swap the provider in the node and the whole graph retargets. This is the exact pattern reused for CRAG and Self-RAG in the RAG notebook.

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_litellm import ChatLiteLLM

graph_llm = ChatLiteLLM(model=MODEL, temperature=0.2, max_tokens=200)

class S(TypedDict):
    question: str
    answer: str

def answer_node(state: S) -> S:
    out = graph_llm.invoke(state["question"])
    return {"answer": out.content}

g = StateGraph(S)
g.add_node("answer", answer_node)
g.add_edge(START, "answer")
g.add_edge("answer", END)
app = g.compile()

out = app.invoke({"question": "In one line, what is a vector database?"})
print(out["answer"])
print("[OK] A LangGraph node wrapped the same LiteLLM model.")

A vector database stores and retrieves high-dimensional numerical representations (embeddings) of data, enabling fast similarity searches based on semantic meaning rather than exact matches.
[OK] A LangGraph node wrapped the same LiteLLM model.


## 13. LiteLLM with AgentCore (deploy, not run here)

The next cell writes an AgentCore entrypoint to disk. It is not executed in the notebook (that would start an HTTP server). The entrypoint code is identical locally and in production; LiteLLM keeps the model swappable even after deploy.

In [ ]:
%%writefile litellm_agentcore_entrypoint.py
# Deploy target: Amazon Bedrock AgentCore Runtime. NOT run inside the notebook.
# Deploy steps:
#   pip install bedrock-agentcore bedrock-agentcore-starter-toolkit strands-agents
#   agentcore configure --entrypoint litellm_agentcore_entrypoint.py
#   agentcore launch
#   then invoke via boto3 client("bedrock-agentcore").invoke_agent_runtime(...)
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models.litellm import LiteLLMModel

model = LiteLLMModel(model_id="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0")
agent = Agent(model=model, system_prompt="You are a concise support agent.")

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    """AgentCore passes the request payload; return a dict."""
    return {"result": str(agent(payload.get("prompt", "Hello")))}

if __name__ == "__main__":
    app.run()   # serves /invocations and /ping on port 8080

Deploy checklist:
1. `pip install bedrock-agentcore bedrock-agentcore-starter-toolkit strands-agents`
2. `agentcore configure --entrypoint litellm_agentcore_entrypoint.py`
3. `agentcore launch` (builds and provisions runtime, IAM, logging)
4. Invoke with `boto3.client("bedrock-agentcore").invoke_agent_runtime(...)`, session id at least 16 chars.

## 14. What LiteLLM is NOT

| People think it is | It is actually | The real thing is |
|---|---|---|
| An agent framework | a model access layer | Strands, LangGraph |
| LangChain | a translation library | LangChain orchestrates, LiteLLM moves one call |
| OpenRouter | self-hosted, your creds | OpenRouter is a hosted paid router |
| A vector database | not storage | FAISS, OpenSearch, Pinecone |
| A caching layer | caching is one feature | Redis |
| A fine-tuning tool | inference-time only | SageMaker, Bedrock Custom Model Import |

One sentence: **LiteLLM is the universal plug between your code and any model. Nothing more, and that is the strength.**

## Recap

- One call, `completion()`, reaches any model by string.
- On Bedrock: `bedrock/us....`, AWS auth not keys, `drop_params=True`.
- Features are one-liners: fallbacks, retries, routing, cost, structured output, streaming, embeddings.
- It is the model layer under Strands, LangChain, LangGraph, AgentCore, never replacing them.

Next: RAG, built directly on the embeddings you just ran.